# End-to-end CKI $^{8}$Be: HFB, Gaussian fidelity, symmetry projection, and non-Gaussianity

This notebook follows one nucleus through the complete workflow:

1. read the CKI nuclear-shell-model interaction;
2. build and diagonalize the exact fixed-$(N,Z)$ Hamiltonian;
3. optimize the intrinsic Bogoliubov/HFB energy;
4. find the pure Gaussian state with the largest ground-state fidelity;
5. apply $P_NP_Z$ and then $P_NP_ZP_{J=0}$;
6. compare energies and exact-ground-state fidelities;
7. align the HFB and fidelity-optimized Gaussian by a spatial rotation;
8. measure best-found geometric non-Gaussianity;
9. inspect how the coherent Euler-vacuum series builds the projected state.

The notebook distinguishes an exact statement from a numerical one: every individual rotated Bogoliubov vacuum is Gaussian, but a coherent sum of such vacua is generally non-Gaussian. Non-Gaussianity is nonlinear and cannot be assigned additively to individual series terms.

## 0. Configuration

The checked-in HFB state is loaded by default so the notebook is reproducible and reasonably quick. Set `RECOMPUTE_HFB=True` to repeat the constrained optimization. Set `FULL_GAUSSIAN_SEARCH=True` for both the finite-Thouless and Slater-boundary searches. The latter is recommended for a publication run but is slower.

In [ ]:
# Particle counts are VALENCE particles outside the inert 4He core.
# 8Be: (2,2), 10Be: (4,2), 12C: (4,4), 14C: (6,4).
VALENCE_NEUTRONS = 2
VALENCE_PROTONS = 2

# None selects the automatically derived finite-space exact grids.
# For default 8Be: NUMBER_GRID=(7,7), EULER_GRID=(9,4,9).
# Larger examples: NUMBER_GRID=(8,8), EULER_GRID=(11,6,11).
NUMBER_GRID = None
EULER_GRID = None

# Set an integer (for example 100) to replace deterministic Euler
# quadrature by Metropolis importance sampling. Number projection stays exact.
METROPOLIS_SAMPLES = None
METROPOLIS_BURN_IN = 1000
METROPOLIS_THINNING = 5
METROPOLIS_SEED = 7

RECOMPUTE_HFB = False
FULL_GAUSSIAN_SEARCH = False
RUN_CUMULATIVE_NONGAUSSIANITY = False

HFB_STARTS = 2
GAUSSIAN_STARTS = 10
CUMULATIVE_STARTS = 2


In [ ]:
from pathlib import Path
from typing import Callable, ClassVar, Dict, List, Optional, Tuple
import itertools
import json
import sys
import time

import numpy as np
from scipy.optimize import minimize

ROOT = Path.cwd()
if not (ROOT / 'src' / 'NSMFermions').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src' / 'NSMFermions'))
sys.path.insert(0, str(ROOT / 'benchmarks'))

from cki_be8 import legacy_definitions, build_fermionic_hamiltonian
from hfb import HFBHamiltonian, HFBState, BogoliubovVacuumSeries, solve_hfb
from number_projection import (
    exact_ground_state, number_projected_series, projected_series_observables,
)
from angular_momentum import (
    ParticleNumberJ0ProjectedEnergy, single_particle_angular_momentum,
    euler_rotation, project_state_observables,
)
from gaussian_fidelity import maximize_gaussian_fidelity, maximize_slater_fidelity


## 1. Read the nucleus and construct the NSM Hamiltonian

The valence-space Hamiltonian is

$$H=\sum_{ij}h_{ij}c_i^\dagger c_j+\frac14\sum_{ijkl}\bar v_{ijkl}c_i^\dagger c_j^\dagger c_lc_k.$$

The CKI model uses an inert $^4$He core and twelve $p$-shell modes: six neutron modes and six proton modes. Therefore the notebook inputs are valence counts, $N_{\rm val}=N_{\rm nucleus}-2$ and $Z_{\rm val}=Z_{\rm nucleus}-2$. For $^{8}$Be, $N_{\rm val}=Z_{\rm val}=2$, and the fixed-$(N,Z)$ space has dimension 225.

In [ ]:
namespace = dict(globals(), trange=range)
legacy_definitions(
    'cg_utils.py',
    ['CG', 'ClebschGordan', 'SelectCG', 'CreateInitialCGList',
     'CalcInitialValues', 'DivCalc', 'CgJM'],
    namespace,
)
legacy_definitions(
    'nuclear_physics_utils.py',
    ['SingleParticleState', 'krond', 'scattering_matrix_reader',
     'compute_nuclear_twobody_matrix', 'get_twobody_nuclearshell_model'],
    namespace,
)

interaction, eps = namespace['get_twobody_nuclearshell_model'](
    str(ROOT / 'data' / 'cki')
)
single_particle = namespace['SingleParticleState'](str(ROOT / 'data' / 'cki'))
state_encoding = single_particle.state_encoding
ham = HFBHamiltonian(np.diag(eps), interaction)
neutron_modes = list(range(6, 12))
proton_modes = list(range(6))
targets = [VALENCE_NEUTRONS, VALENCE_PROTONS]
if any(number < 0 or number > 6 for number in targets):
    raise ValueError('CKI has only six valence modes for each species')
if sum(targets) == 0:
    raise ValueError('This tutorial expects at least one valence particle')
if sum(targets) % 2:
    raise ValueError(
        'The present HFB vacuum and J=0 projector require even total valence number; '
        'odd systems need blocked HFB and a J>0 projection workflow.'
    )
fermionic = build_fermionic_hamiltonian(
    interaction, eps, particles=tuple(targets)
)
exact_energy, exact_target = exact_ground_state(fermionic)

print('valence (N,Z):', tuple(targets))
print('nuclear (N,Z) with the 4He core:',
      (targets[0] + 2, targets[1] + 2))
print('modes:', len(eps))
print('fixed-(N,Z) dimension:', len(fermionic.occupations))
print('exact ground-state energy:', exact_energy, 'MeV')


## 2. Optimize or load the intrinsic HFB state

HFB minimizes

$$E[\rho,\kappa]=\langle\Phi|H|\Phi\rangle$$

subject to $\langle N\rangle=N_{\rm val}$ and $\langle Z\rangle=Z_{\rm val}$. The returned `HFBResult` contains the state, energy, particle numbers, convergence flag, optimizer attempts, and stationarity diagnostics.

For this bounded CKI run, pairing collapses numerically: $\|\kappa\|\sim10^{-5}$. A nearly singular-$U$ state lies between the finite-$Z$ and exact-Slater numerical charts, so the helper below replaces it by the corresponding occupied natural-orbital Slater determinant when the collapse diagnostics are small. This changes neither the physical interpretation nor the quoted accuracy, and makes projection robust.

In [ ]:
use_saved_be8 = (tuple(targets) == (2, 2) and not RECOMPUTE_HFB)
if not use_saved_be8:
    hfb_result = solve_hfb(
        ham, neutron_modes, targets, starts=HFB_STARTS, seed=8,
        maxiter=120, tolerance=1e-8,
    )
    hfb_raw = hfb_result.state
    print('optimizer converged:', hfb_result.converged)
    print('optimizer attempts:', hfb_result.attempts)
else:
    saved = np.load(ROOT / 'benchmarks' / 'results' / 'cki_be8_state.npz')
    hfb_raw = HFBState(saved['U'], saved['V'])

rho_hfb = (hfb_raw.rho + hfb_raw.rho.conj().T) / 2
rho_idempotency = np.linalg.norm(rho_hfb @ rho_hfb - rho_hfb)
pairing_norm = np.linalg.norm(hfb_raw.kappa)
occupations_rho, natural_orbitals = np.linalg.eigh(rho_hfb)
hfb_orbitals = None
if pairing_norm < 1e-4 and rho_idempotency < 1e-6:
    # Use the exact Slater chart only when the optimized state has collapsed.
    hfb_orbitals = natural_orbitals[:, -sum(targets):]
    hfb_state = HFBState.from_slater(hfb_orbitals)
    hfb_chart = 'Slater (pairing collapsed)'
else:
    # Retain genuine pairing for nuclei whose HFB minimum is not a Slater state.
    hfb_state = hfb_raw
    hfb_chart = 'paired Bogoliubov vacuum'

print('raw HFB energy:', ham.energy(hfb_raw), 'MeV')
print('state used below:', hfb_chart)
print('working-state energy:', ham.energy(hfb_state), 'MeV')
print('||kappa||:', pairing_norm)
print('||rho^2-rho||:', rho_idempotency)
print('working-state canonical error:', hfb_state.canonical_error())


The intrinsic fidelity is the full-Fock-space quantity

$$F_{\rm HFB}=|\langle\Psi_0|\Phi_{\rm HFB}\rangle|^2.$$

Because $|\Psi_0\rangle$ has fixed $N,Z$, this includes the probability that the intrinsic state lies in that sector.

In [ ]:
hfb_fidelity = hfb_state.fixed_sector_fidelity(
    exact_target, fermionic.occupations
)
hfb_NZ_weight = hfb_state.fixed_sector_weight(fermionic.occupations)
print('HFB N,Z weight:', hfb_NZ_weight)
print('HFB ground-state fidelity:', hfb_fidelity)


## 3. Find the closest pure Gaussian to the exact ground state

The geometric Gaussian fidelity is

$$F_G(\Psi_0)=\max_{|\Omega\rangle\in\mathcal G}|\langle\Psi_0|\Omega\rangle|^2,$$

and the geometric non-Gaussianity is

$$\mathcal N_G(\Psi_0)=1-F_G(\Psi_0).$$

The even Gaussian manifold has a finite-Thouless interior and a singular number-conserving Slater boundary. A complete numerical search must examine both. For CKI $^{8}$Be, repeated Slater starts converge to the same value and outperform the finite-$Z$ search, providing strong numerical evidence—not a mathematical global-optimum certificate—that the closest Gaussian is a Slater determinant.

In [ ]:
slater_best = maximize_slater_fidelity(
    fermionic, exact_target, starts=GAUSSIAN_STARTS, seed=42,
    maxiter=1500, gradient_tolerance=2e-7,
    initial_orbitals=hfb_orbitals,
)
gaussian_candidates = [('Slater boundary', slater_best.fidelity)]
interior_best = None
if FULL_GAUSSIAN_SEARCH:
    interior_best = maximize_gaussian_fidelity(
        fermionic, exact_target, starts=16, seed=41, maxiter=2000,
        tolerance=1e-15, gradient_tolerance=2e-6,
    )
    gaussian_candidates.append(('finite-Z interior', interior_best.fidelity))

best_kind, exact_best_gaussian_fidelity = max(
    gaussian_candidates, key=lambda item: item[1]
)
if best_kind == 'Slater boundary':
    best_gaussian_state = HFBState.from_slater(slater_best.orbitals)
else:
    best_gaussian_state = interior_best.state

print('candidates:', gaussian_candidates)
print('selected:', best_kind)
print('best-found Gaussian fidelity:', exact_best_gaussian_fidelity)
print('best-found ground-state non-Gaussianity:',
      1 - exact_best_gaussian_fidelity)


When `FULL_GAUSSIAN_SEARCH=False`, the displayed value is the optimized Slater-boundary result. Set it to `True` before interpreting the result as the best found over both implemented charts.

## 4. Particle-number projection

The double Fourier projector is

$$P_NP_Z=\frac1{(2\pi)^2}\int d\varphi_Nd\varphi_Z\,e^{i\varphi_N(\hat N-N)}e^{i\varphi_Z(\hat Z-Z)}.$$

In the finite space, the integrals are exact discrete sums. The projected ket remains a coherent `BogoliubovVacuumSeries` until observables require determinant amplitudes.

In [ ]:
pn_series_hfb = number_projected_series(
    hfb_state, fermionic, grid=NUMBER_GRID
)
pn_hfb = projected_series_observables(pn_series_hfb, fermionic, exact_target)
pn_series_gaussian = number_projected_series(
    best_gaussian_state, fermionic, grid=NUMBER_GRID
)
pn_gaussian = projected_series_observables(
    pn_series_gaussian, fermionic, exact_target
)

print('number-grid vacua:', pn_series_hfb.number_of_vacua)
print('P_N P_Z HFB energy/fidelity:', pn_hfb.energy, pn_hfb.fidelity)
print('P_N P_Z best-Gaussian energy/fidelity:',
      pn_gaussian.energy, pn_gaussian.fidelity)


For a number-conserving Slater determinant with no neutron-proton mixing, $P_NP_Z$ changes nothing. More generally, write $|\Phi\rangle=\sum_{N,Z}c_{NZ}|\Phi_{NZ}\rangle$ and $w_{NZ}=|c_{NZ}|^2$. For a normalized exact target in the selected sector, $F_{\rm raw}=w_{NZ}F_{\rm projected}$. In the default run, pairing has collapsed and `HFB N,Z weight` is $0.9999999996$, so number projection leaves the energy and fidelity unchanged to numerical precision. A genuinely paired HFB state has $w_{NZ}<1$ and number projection generally changes both quantities.

## 5. Simultaneous number and $J=0$ projection

For the $0^+$ ground state,

$$P_0=\frac1{8\pi^2}\int d\alpha\,d\gamma\,d(\cos\beta)\;R(\alpha,\beta,\gamma).$$

The CKI bounds give a $9\times4\times9$ Euler grid. Combined with the $7\times7$ number grid, each deterministic projected series contains 15,876 Gaussian vacua. Alternatively, Metropolis importance sampling replaces only the Euler grid: with $S$ retained rotations the series contains $7\times7\times S$ vacua. Every sampled rotation still carries the complete number grid, so $N,Z$ projection remains exact while $J=0$ projection becomes stochastic.

In [ ]:
j0_projector = ParticleNumberJ0ProjectedEnergy(
    ham, state_encoding, neutron_modes, targets,
    number_grid=NUMBER_GRID, euler_grid=EULER_GRID,
)
def build_pnj_series(state, seed):
    if METROPOLIS_SAMPLES is None:
        return j0_projector.projected_series(state)
    return j0_projector.metropolis_projected_series(
        state, METROPOLIS_SAMPLES,
        burn_in=METROPOLIS_BURN_IN,
        thinning=METROPOLIS_THINNING,
        seed=seed,
    )

pnj_series_hfb = build_pnj_series(hfb_state, METROPOLIS_SEED)
pnj_hfb = project_state_observables(pnj_series_hfb, fermionic, exact_target)
pnj_series_gaussian = build_pnj_series(
    best_gaussian_state, METROPOLIS_SEED
)
pnj_gaussian = project_state_observables(
    pnj_series_gaussian, fermionic, exact_target
)

print('sampling method:', pnj_series_hfb.sampling_method)
print('Euler grid/samples:', pnj_series_hfb.euler_grid)
print('total vacua:', pnj_series_hfb.number_of_vacua)
if pnj_series_hfb.sampling_diagnostics is not None:
    print('HFB Metropolis diagnostics:',
          pnj_series_hfb.sampling_diagnostics)
print('P_N P_Z P_J=0 HFB energy/fidelity:',
      pnj_hfb.energy, pnj_hfb.fidelity)
print('P_N P_Z P_J=0 best-Gaussian energy/fidelity:',
      pnj_gaussian.energy, pnj_gaussian.fidelity)


## 6. Compare every state with the exact ground state

In [ ]:
rows = [
    ('Exact ground state', exact_energy, 1.0, 1),
    ('Intrinsic HFB/Slater', ham.energy(hfb_state), hfb_fidelity, 1),
    ('Closest Gaussian', ham.energy(best_gaussian_state),
     exact_best_gaussian_fidelity, 1),
    ('P_N P_Z HFB', pn_hfb.energy, pn_hfb.fidelity,
     pn_series_hfb.number_of_vacua),
    ('P_N P_Z closest Gaussian', pn_gaussian.energy, pn_gaussian.fidelity,
     pn_series_gaussian.number_of_vacua),
    ('P_N P_Z P_J=0 HFB', pnj_hfb.energy, pnj_hfb.fidelity,
     pnj_series_hfb.number_of_vacua),
    ('P_N P_Z P_J=0 closest Gaussian', pnj_gaussian.energy,
     pnj_gaussian.fidelity, pnj_series_gaussian.number_of_vacua),
]
print(f"{'state':38s} {'energy (MeV)':>15s} {'F(exact)':>14s} {'vacua':>9s}")
print('-' * 80)
for name, energy, fidelity, vacua in rows:
    print(f'{name:38s} {energy:15.9f} {fidelity:14.9f} {vacua:9d}')


With the checked-in HFB state and converged Slater-boundary search, the reference run gives $F_{\rm HFB}=0.2018135$, $F_{G}=0.2021472$, $F_{NZJ=0}^{\rm HFB}=0.9954831$, and $F_{NZJ=0}^{G}=0.9865244$. Number projection alone barely changes either nearly number-conserving Slater state.

## 7. Are the HFB and closest-Gaussian states related by a rotation?

For Slater orbital matrices $C_H,C_G$,

$$F(\Omega)=|\det(C_H^\dagger R(\Omega)C_G)|^2.$$

The raw overlap can be nearly zero even when the two occupied subspaces differ almost entirely by orientation. The relevant test is $\max_\Omega F(\Omega)$.

In [ ]:
if hfb_orbitals is not None and best_kind == 'Slater boundary':
    gaussian_orbitals = slater_best.orbitals
    generators = single_particle_angular_momentum(state_encoding)

    def slater_overlap_fidelity(left, right):
        return float(abs(np.linalg.det(left.conj().T @ right))**2)

    def rotated_fidelity(angles):
        rotation = euler_rotation(*angles, generators)
        return slater_overlap_fidelity(
            hfb_orbitals, rotation @ gaussian_orbitals
        )

    rng = np.random.default_rng(18)
    fits = [
        minimize(lambda angles: -rotated_fidelity(angles),
                 rng.uniform(-np.pi, np.pi, 3), method='BFGS',
                 options={'maxiter': 300, 'gtol': 1e-10})
        for _ in range(30)
    ]
    alignment = min(fits, key=lambda fit: fit.fun)
    raw_mutual_fidelity = slater_overlap_fidelity(
        hfb_orbitals, gaussian_orbitals
    )
    aligned_mutual_fidelity = float(-alignment.fun)
    print('raw HFB/Gaussian fidelity:', raw_mutual_fidelity)
    print('best spatially aligned fidelity:', aligned_mutual_fidelity)
    print('Euler angles (radians):', alignment.x)
else:
    raw_mutual_fidelity = aligned_mutual_fidelity = None
    print('The determinant alignment formula applies when both selected')
    print('states are Slater determinants. A paired-state cross-Pfaffian')
    print('alignment is not implemented in this tutorial.')


A value near one after alignment means the two intrinsic states are almost the same deformed Slater shape in different laboratory orientations. It does not mean the exact $J=0$ ground state is Gaussian.

## 8. Best-found non-Gaussianity of the $J=0$-projected state

For any normalized fixed-sector target $|\psi\rangle$, the implemented estimate is

$$\mathcal N_G^{\rm best\ found}(|\psi\rangle)=1-\max(F_{\rm finite-Z},F_{\rm Slater}).$$

The optimization is non-convex, so this is a reproducible best-found value, not a certified global optimum. The helper searches the Slater boundary always and the finite-$Z$ chart when `FULL_GAUSSIAN_SEARCH=True`.

In [ ]:
def best_found_gaussian_fidelity(target_vector, starts=GAUSSIAN_STARTS):
    boundary = maximize_slater_fidelity(
        fermionic, target_vector, starts=starts, seed=52,
        maxiter=1500, gradient_tolerance=2e-7,
    )
    candidates = {'Slater boundary': boundary.fidelity}
    interior = None
    if FULL_GAUSSIAN_SEARCH:
        interior = maximize_gaussian_fidelity(
            fermionic, target_vector, starts=max(4, starts), seed=53,
            maxiter=1500, gradient_tolerance=2e-6,
        )
        candidates['finite-Z interior'] = interior.fidelity
    kind = max(candidates, key=candidates.get)
    return candidates[kind], 1-candidates[kind], kind, candidates

j0_best_fidelity, j0_nongaussianity, j0_best_kind, j0_candidates = (
    best_found_gaussian_fidelity(pnj_hfb.projected_vector)
)
print('J=0 projected candidates:', j0_candidates)
print('selected chart:', j0_best_kind)
print('best-found Gaussian fidelity:', j0_best_fidelity)
print('best-found non-Gaussianity:', j0_nongaussianity)


The intrinsic HFB and closest-Gaussian states have zero non-Gaussianity by construction. A number projection of an exactly number-conserving Slater state also leaves it Gaussian. Angular-momentum projection creates a coherent superposition of differently oriented Slater determinants and can therefore generate substantial non-Gaussianity.

## 9. Resolve the $J=0$ series into Euler-group contributions

Write the projected state as

$$|\Psi\rangle=\sum_{r=1}^{L_\Omega}|C_r\rangle,\qquad |C_r\rangle=\sum_{n,z}w_{nzr}G_{nz}R_r|\Phi\rangle.$$

Each individual $G_{nz}R_r|\Phi\rangle$ is Gaussian. Each $|C_r\rangle$ already contains the complete number projector and is generally non-Gaussian. The vectors interfere, so neither norms nor non-Gaussianity add term by term. We can nevertheless inspect cumulative normalized states

$$|\Psi_K\rangle=\frac{\sum_{r=1}^K|C_r\rangle}{\|\sum_{r=1}^K|C_r\rangle\|}.$$

This is a convergence diagnostic, not a unique decomposition: it depends on the chosen Euler-node order.

In [ ]:
def slater_series_components_by_euler(series, orbitals, occupations):
    occupations = np.asarray(occupations, dtype=int)
    euler_points = int(np.prod(series.euler_grid))
    number_points = int(np.prod(series.number_grid))
    if series.sampling_method == 'quadrature':
        transforms = series.transformations.reshape(
            number_points, euler_points, len(orbitals), len(orbitals)
        )
        weights = series.weights.reshape(number_points, euler_points)
    else:
        # Metropolis stores one rotation followed by its complete number grid.
        transforms = series.transformations.reshape(
            euler_points, number_points, len(orbitals), len(orbitals)
        ).transpose(1, 0, 2, 3)
        weights = series.weights.reshape(
            euler_points, number_points
        ).T
    components = np.zeros((euler_points, len(occupations)), complex)
    for number_index in range(number_points):
        for euler_index in range(euler_points):
            rotated = transforms[number_index, euler_index] @ orbitals
            components[euler_index] += (
                weights[number_index, euler_index]
                * np.linalg.det(rotated[occupations])
            )
    return components

def generic_series_components_by_euler(series, occupations):
    euler_points = int(np.prod(series.euler_grid))
    components = []
    for euler_index in range(euler_points):
        if series.sampling_method == 'quadrature':
            indices = np.arange(
                euler_index, series.number_of_vacua, euler_points
            )
        else:
            number_points = int(np.prod(series.number_grid))
            start = euler_index * number_points
            indices = np.arange(start, start + number_points)
        contribution = BogoliubovVacuumSeries(
            intrinsic_state=series.intrinsic_state,
            transformations=series.transformations[indices],
            weights=series.weights[indices],
            number_grid=series.number_grid,
            euler_grid=(1,),
            projection='one Euler group with exact P_N P_Z',
            number_offset=series.number_offset,
        )
        components.append(contribution.occupation_amplitudes(occupations))
    return np.asarray(components)

if hfb_orbitals is not None:
    euler_components = slater_series_components_by_euler(
        pnj_series_hfb, hfb_orbitals, fermionic.occupations
    )
else:
    euler_components = generic_series_components_by_euler(
        pnj_series_hfb, fermionic.occupations
    )
cumulative = np.cumsum(euler_components, axis=0)
checkpoint_candidates = [1, 2, 4, 8, 16, 32, 64, 128, len(cumulative)]
checkpoints = sorted(set(
    count for count in checkpoint_candidates if count <= len(cumulative)
))
cumulative_rows = []
for count in checkpoints:
    vector = cumulative[count-1]
    norm = float(np.vdot(vector, vector).real)
    normalized = vector / np.sqrt(norm)
    exact_fidelity = float(abs(np.vdot(exact_target, normalized))**2)
    cumulative_rows.append((count, norm, exact_fidelity, normalized))

print(f"{'Euler groups K':>14s} {'unnormalized norm':>20s} {'F(exact)':>14s}")
for count, norm, fidelity, _ in cumulative_rows:
    print(f'{count:14d} {norm:20.10e} {fidelity:14.9f}')
full_cumulative = cumulative[-1] / np.linalg.norm(cumulative[-1])
full_series_match = float(
    abs(np.vdot(pnj_hfb.projected_vector, full_cumulative))**2
)
print('full-series projective fidelity:', full_series_match)


The unnormalized norm is not monotonic because complex amplitudes interfere. The final checkpoint must reproduce the full projected state.

In [ ]:
if RUN_CUMULATIVE_NONGAUSSIANITY:
    print(f"{'K':>6s} {'best Gaussian F':>18s} {'N_G':>14s}")
    # Sparse checkpoints keep the repeated non-convex optimizations manageable.
    selected = cumulative_rows[::max(1, len(cumulative_rows)//4)]
    if selected[-1][0] != cumulative_rows[-1][0]:
        selected.append(cumulative_rows[-1])
    for count, _, _, vector in selected:
        best_f, non_g, _, _ = best_found_gaussian_fidelity(
            vector, starts=CUMULATIVE_STARTS
        )
        print(f'{count:6d} {best_f:18.10f} {non_g:14.10f}')
else:
    print('Set RUN_CUMULATIVE_NONGAUSSIANITY=True to run the optional')
    print('ordering-dependent Gaussian optimization at sparse Euler checkpoints.')


## Interpretation

- HFB energy optimization and Gaussian-fidelity optimization solve different variational problems.
- Similar exact-ground-state fidelities do not imply a large mutual overlap.
- For a $J=0$ target, all spatial orientations of the same intrinsic Gaussian have identical target fidelity.
- Rotation alignment tests whether two intrinsic solutions lie on approximately the same symmetry orbit.
- Number and angular-momentum projection restore symmetries by coherent group averaging.
- A single rotated vacuum has $\mathcal N_G=0$, while the symmetry-restored coherent sum may have $\mathcal N_G>0$.
- There is no basis-independent additive 'non-Gaussianity of term $q$'. Cumulative values are useful convergence diagnostics but depend on term grouping and ordering.